Stage 1: Data analysis and data understanding

In [14]:
#import bib for data analysis
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder
from webencodings import labels
from sklearn.pipeline import Pipeline

from titanic.understand_data import y_pred


1.1 : observation

In [2]:
df = pd.read_csv('train.csv')
print(
    'number of cols in train dataset: \n',
    len(df.columns),#df.shape[1]
    '\n------------------------------------------------\n',
    'number of categorial cols in train dataset: \n',
    df.select_dtypes(exclude=['number']).columns,len(df.select_dtypes(exclude=['number']).columns),
    '\n------------------------------------------------\n',
    'number of numerical cols in train dataset: \n',
    df.select_dtypes(include=['number']).columns,len(df.select_dtypes(include=['number']).columns),
    '\n------------------------------------------------\n',
    'number of data row in the dataset: \n',
    df.shape[0]#len(df)
)

number of cols in train dataset: 
 15 
------------------------------------------------
 number of categorial cols in train dataset: 
 Index(['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible',
       'Subsidy_Available', 'Range_Anxiety_Level', 'Will_Buy_EV'],
      dtype='str') 7 
------------------------------------------------
 number of numerical cols in train dataset: 
 Index(['id', 'Age', 'Annual_Income_USD', 'Daily_Commute_km',
       'Number_of_Cars_Owned', 'Charging_Stations_Near_Home',
       'Charging_Stations_Near_Work', 'Environmental_Concern_Level'],
      dtype='str') 8 
------------------------------------------------
 number of data row in the dataset: 
 668665


from here i undestand many thing altough i didn't finish my observation,
first , the dataset is too long ,usually kaggle put at maximum 10'000 but now they put more then half of a million so i cauld say that they prerotize efficiency
second , we need to remember that the goal is to predict Electric Vehicle Interest so the model will be progression model because we have continue prediction
next, i will see some general info

In [3]:
#df.set_index('id', inplace=True)
df.head()

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
0,0,66,92887.0,23.4,2,3,7,1.0,Male,Suburban,Sedan,Yes,No,Low,No
1,1,38,30000.0,5.0,1,2,2,4.0,Male,Rural,SUV,Yes,No,Low,No
2,2,26,94389.0,36.8,1,8,15,5.0,Female,Urban,Sedan,No,Yes,Low,Yes
3,3,66,73580.0,23.7,2,6,9,3.0,Male,Suburban,Hatchback,Yes,No,Low,No
4,4,54,57898.0,50.8,1,2,3,3.0,Male,Suburban,Hatchback,Yes,No,Low,No


In [4]:
#let's see more about the dataset
print(
    '----------------------------------\n',
    df.info(),
    '\n------------------------------\n',
    df.isna().sum(),
    '\n------------------------------\n',
)

<class 'pandas.DataFrame'>
RangeIndex: 668665 entries, 0 to 668664
Data columns (total 15 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   id                           668665 non-null  int64  
 1   Age                          668665 non-null  int64  
 2   Annual_Income_USD            668665 non-null  float64
 3   Daily_Commute_km             668665 non-null  float64
 4   Number_of_Cars_Owned         668665 non-null  int64  
 5   Charging_Stations_Near_Home  668665 non-null  int64  
 6   Charging_Stations_Near_Work  668665 non-null  int64  
 7   Environmental_Concern_Level  668665 non-null  float64
 8   Gender                       668665 non-null  str    
 9   City_Type                    668665 non-null  str    
 10  Current_Car_Type             668665 non-null  str    
 11  Home_Charging_Possible       668665 non-null  str    
 12  Subsidy_Available            668665 non-null  str    
 13  Range_Anxi

wow this so unusual from kaggle the dataset doesn't has any null value , let's check something else

In [5]:
df.describe()

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level
count,668665.000000,668665.000000,668665.000000,668665.000000,668665.000000,668665.000000,668665.000000,668665.000000
mean,334332.000000,47.039171,84769.266989,32.158298,1.712626,4.960408,7.176314,2.935477
std,193027.103211,12.875448,28648.029042,18.730474,0.729275,3.926843,5.186627,1.429119
min,0.000000,25.000000,30000.000000,5.000000,1.000000,0.000000,0.000000,1.000000
25%,167166.000000,36.000000,67376.000000,17.200000,1.000000,2.000000,3.000000,2.000000
50%,334332.000000,47.000000,84880.000000,33.600000,2.000000,4.000000,6.000000,3.000000
75%,501498.000000,58.000000,102753.000000,47.400000,2.000000,7.000000,10.000000,4.000000
max,668664.000000,69.000000,188549.000000,98.700000,4.000000,14.000000,19.000000,5.000000


In [6]:
for i in df.select_dtypes(exclude=['number']).columns.values:
    print(df[i].unique())

<StringArray>
['Male', 'Female', 'Other']
Length: 3, dtype: str
<StringArray>
['Suburban', 'Rural', 'Urban']
Length: 3, dtype: str
<StringArray>
['Sedan', 'SUV', 'Hatchback', 'Truck']
Length: 4, dtype: str
<StringArray>
['Yes', 'No']
Length: 2, dtype: str
<StringArray>
['No', 'Yes']
Length: 2, dtype: str
<StringArray>
['Low', 'Medium', 'High']
Length: 3, dtype: str
<StringArray>
['No', 'Yes']
Length: 2, dtype: str


In [7]:
for i in df.columns.values: #df.apply(lamda x: x.unique())
    print('----------------------------------------------------\n')
    print(i)
    print(df[i].unique(),'nunique:',df[i].nunique())

----------------------------------------------------

id
[     0      1      2 ... 668662 668663 668664] nunique: 668665
----------------------------------------------------

Age
[66 38 26 54 50 55 45 39 57 25 65 59 36 51 41 32 48 46 60 28 68 56 49 67
 43 29 42 31 58 35 53 37 34 47 52 69 63 64 40 30 62 61 33 44 27] nunique: 45
----------------------------------------------------

Annual_Income_USD
[ 92887.  30000.  94389. ... 123813. 121432.  86857.] nunique: 13214
----------------------------------------------------

Daily_Commute_km
[23.4  5.  36.8 23.7 50.8 55.1 42.9 52.2 31.5 66.9 27.6 47.4 33.3 19.5
 44.5 35.4 76.4 25.4 18.6 36.2 49.4 20.1 42.7 32.6 54.8 25.8 50.6 48.
 32.  31.  29.8 13.2 45.4 41.3 37.7 21.  11.2 45.9 28.5 39.6 30.4 34.4
 57.7 52.1 59.  22.  33.4 47.1 37.2 33.6 26.5 54.3 53.2 61.1 47.2 37.5
 57.2 52.  12.8 28.3 57.1 12.7 41.2 48.7 45.8 43.9 37.1 39.3 51.2 64.6
 49.2 36.  47.9 43.6 49.3 48.9 49.1 11.7 43.3 18.8 27.9 30.9 24.4 22.4
 35.7 66.6 58.9 27.5 30.1 49.  28.

now let twick our data a little bit more to make the work of the model a little bit easier, as you can see we change some col to bool

In [60]:
def clean_data_pipeline(link):
    return (pd.read_csv(link).pipe(lambda x: x.set_index('id'))
 .pipe(lambda x: x.assign(Gender = [True if i == 'Male' else False for i in x['Gender']]))
 .pipe(lambda x: x.assign(Home_Charging_Possible =[True if i == 'Yes' else False for i in x['Home_Charging_Possible']] ))
 .pipe(lambda x: x.assign(Subsidy_Available = [True if i == 'Yes' else False for i in x['Subsidy_Available']] ))
 #.pipe(lambda x: x.assign(Will_Buy_EV = [True if i == 'Yes' else False for i in x['Will_Buy_EV']]))
 .pipe(lambda x: x.assign(City_Type = x['City_Type'].str[0])))

In [9]:
df = clean_data_pipeline('train.csv')
df.head()

,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
id,,,,,,,,,,,,,,
0,66,92887.0,23.4,2,3,7,1.0,True,S,Sedan,True,False,Low,False
1,38,30000.0,5.0,1,2,2,4.0,True,R,SUV,True,False,Low,False
2,26,94389.0,36.8,1,8,15,5.0,False,U,Sedan,False,True,Low,True
3,66,73580.0,23.7,2,6,9,3.0,True,S,Hatchback,True,False,Low,False
4,54,57898.0,50.8,1,2,3,3.0,True,S,Hatchback,True,False,Low,False


ok i guess we are ready to start by the model creation , i hope we can do good

In [71]:
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold

X = df.drop('Will_Buy_EV', axis=1)#.iloc[0:28000]
y = df['Will_Buy_EV']#.iloc[0:28000]
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=7)
cat = X.select_dtypes(exclude=['number','bool']).columns.values
num = X.select_dtypes(include=['number']).columns.values
boolC = X.select_dtypes(include=['bool']).columns.values

In [12]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder


models = {
'LR':LogisticRegression(),
'DTC':DecisionTreeClassifier(),
'RFC':RandomForestClassifier(),
}

preprocessor = ColumnTransformer(
    transformers=[
        ('cat',OneHotEncoder(handle_unknown='ignore'),cat)
    ]
)

result = []
kf = KFold(n_splits=5, random_state=7 ,shuffle=True)
for i in models:
    model = Pipeline(steps=[('preprocessor',preprocessor),
                            ('model',models[i])])
    cv_results = cross_val_score(model, X_train, y_train, cv=kf)
    result.append(cv_results)
print(result)
plt.boxplot(result, tick_labels=models.keys())
plt.show()


KeyboardInterrupt: 

i think i will choose LogisticRegression or one of his decendent (lasso or ridge)

In [72]:
from sklearn.preprocessing import StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ('cat',OneHotEncoder(handle_unknown='ignore'),cat),
        ('num', StandardScaler(), num),
         ('bool', 'passthrough', boolC)
    ]
)
model = Pipeline(steps=[('preprocessor',preprocessor),('model',LogisticRegression(max_iter=1000))])
model.fit(X_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[bool](2,)","[False, True]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](13,)","['Age','Annual_Income_USD','Daily_Commute_km',...,'Home_Charging_Possible', 'Subsidy_Available','Range_Anxiety_Level']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,13
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of 

In [51]:

y_pred = model.predict_proba(X_test)
print(y_pred[:,1])

[0.17893895 0.20544903 0.17893895 ... 0.1886138  0.03924386 0.04144969]


In [74]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
y_pred = model.predict(X_test)
y_pred = (y_pred >= 0.5).astype(int)
cm = confusion_matrix(y_test,y_pred)
classification_report(y_test,y_pred)
print('accuracy:',accuracy_score(y_test,y_pred))
print(cm)
print(classification_report(y_test,y_pred))

accuracy: 0.8946557693314291
[[104178   6071]
 [  8017  15467]]
              precision    recall  f1-score   support

       False       0.93      0.94      0.94    110249
        True       0.72      0.66      0.69     23484

    accuracy                           0.89    133733
   macro avg       0.82      0.80      0.81    133733
weighted avg       0.89      0.89      0.89    133733



In [85]:
X_test_sub = clean_data_pipeline('test.csv')
y_pred_proba = grid.predict_proba(X_test_sub)

sub = pd.DataFrame(
    {
        'id':X_test_sub.index,
        'Will_Buy_EV':y_pred_proba[:,1],
    }
)
sub.to_csv('submission.csv',index=False)


In [68]:
logr = model.named_steps['model']
logr.coef_[0]

array([-0.52888799, -0.5783676 , -0.6428778 , -0.41322822, -0.37304671,
       -0.43003051, -0.53382794, -2.70628893,  1.31526412, -0.35910858])

In [83]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
cv = StratifiedKFold(n_splits=5, random_state=260422 ,shuffle=True)
score = cross_val_score(grid, X, y, cv=cv,scoring='roc_auc',n_jobs=-1)

In [84]:
print(score)
print('mean score:',score.mean())

[0.9382647  0.93814572 0.93789392 0.9375341  0.9386498 ]
mean score: 0.938097647352647


In [82]:
from sklearn.model_selection import GridSearchCV
params ={
    'model__C': [0.001, 0.01, 0.1, 0.3, 1, 3, 10, 30, 100]
}
grid = GridSearchCV(
    model,
    param_grid=params,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
)
grid.fit(X,y)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step..._iter=1000))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__C': [0.001, 0.01, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, default=0Controls the verbosity of inform

In [ ]:
from sklearn.svm import SV